# Supervised Learning — AI Engineering Interview Prep

Regression + Classification: key algorithms side-by-side using sklearn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded.")

## 1. Classification — Breast Cancer Dataset

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=200),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5),
    'Random Forest':       RandomForestClassifier(n_estimators=100),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100),
    'SVM (RBF)':           SVC(kernel='rbf'),
    'KNN (k=5)':           KNeighborsClassifier(n_neighbors=5),
}

results = []
for name, clf in classifiers.items():
    use_scaled = name in ['Logistic Regression', 'SVM (RBF)', 'KNN (k=5)']
    Xt, Xv = (X_train_s, X_test_s) if use_scaled else (X_train, X_test)
    clf.fit(Xt, y_train)
    train_acc = clf.score(Xt, y_train)
    test_acc  = clf.score(Xv, y_test)
    cv_scores = cross_val_score(clf, Xt, y_train, cv=5, scoring='accuracy')
    results.append({'Model': name, 'Train Acc': train_acc, 'Test Acc': test_acc,
                    'CV Mean': cv_scores.mean(), 'CV Std': cv_scores.std()})

results_df = pd.DataFrame(results).sort_values('Test Acc', ascending=False)
print(results_df.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# Feature importance from Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
importances = pd.Series(rf.feature_importances_, index=data.feature_names)

plt.figure(figsize=(10, 6))
importances.nlargest(15).sort_values().plot(kind='barh', color='steelblue')
plt.title("Top 15 Feature Importances (Random Forest)")
plt.tight_layout()
plt.show()

## 2. Regression — California Housing

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

housing = fetch_california_housing()
X_h, y_h = housing.data, housing.target
X_tr, X_te, y_tr, y_te = train_test_split(X_h, y_h, test_size=0.2, random_state=42)

scaler_h = StandardScaler()
X_tr_s = scaler_h.fit_transform(X_tr)
X_te_s = scaler_h.transform(X_te)

regressors = {
    'Linear Regression': (LinearRegression(), True),
    'Ridge (α=1)':        (Ridge(alpha=1.0), True),
    'Lasso (α=0.01)':     (Lasso(alpha=0.01), True),
    'Random Forest':      (RandomForestRegressor(n_estimators=100, random_state=42), False),
    'Gradient Boosting':  (GradientBoostingRegressor(n_estimators=100, random_state=42), False),
}

reg_results = []
for name, (reg, use_scale) in regressors.items():
    Xt, Xv = (X_tr_s, X_te_s) if use_scale else (X_tr, X_te)
    reg.fit(Xt, y_tr)
    y_pred = reg.predict(Xv)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    r2   = r2_score(y_te, y_pred)
    reg_results.append({'Model': name, 'RMSE': rmse, 'R²': r2})

pd.DataFrame(reg_results).sort_values('RMSE').to_string(index=False, float_format='{:.4f}'.format)

## 3. Bias-Variance Tradeoff

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

X_1d = np.sort(np.random.uniform(0, 10, 60))
y_1d = np.sin(X_1d) + np.random.normal(0, 0.3, 60)
X_1d_r = X_1d.reshape(-1, 1)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, degree in zip(axes, [1, 3, 10, 20]):
    pipe = Pipeline([('poly', PolynomialFeatures(degree)), ('lr', LinearRegression())])
    pipe.fit(X_1d_r, y_1d)
    x_plot = np.linspace(0, 10, 200).reshape(-1, 1)
    ax.scatter(X_1d, y_1d, s=15, alpha=0.5)
    ax.plot(x_plot, pipe.predict(x_plot), 'r-', lw=2)
    train_r2 = pipe.score(X_1d_r, y_1d)
    label = 'underfitting' if degree==1 else ('ok' if degree<=5 else 'overfitting')
    ax.set_title(f'Degree={degree}\nR²={train_r2:.2f} ({label})')

plt.suptitle("Bias-Variance Tradeoff", y=1.02)
plt.tight_layout()
plt.show()

## Interview Tips

- **Logistic Regression**: excellent baseline for classification; fast, interpretable, needs scaling.
- **Random Forest vs GBM**: RF is parallel (trees in parallel), robust to noise; GBM sequential, higher accuracy but slower and needs more tuning.
- **Ridge (L2)**: shrinks all coefficients, keeps all features. **Lasso (L1)**: drives some to zero (feature selection).
- **SVM**: powerful with kernel trick but slow on large datasets (O(n³) training). Scale features!
- **Bias-variance**: high bias = underfitting (too simple); high variance = overfitting (memorises training set).
- **Tree depth**: max_depth controls variance. Deeper = lower bias but higher variance.